# Tutorial 9: Capstone — your own pipeline, end to end

Estimated time: 60-90 minutes

## Prerequisites
**PyMC installed.** Use the `py312_bayesmm_pymc` kernel, or `py312_bayesmm_all` if you are
working through the whole series. Note that `py312_bayesmm_sbi` is deliberately sbi-only
since sbi 0.26 (its own `environment-sbi.yml` header explains why) and will **not** work
here. Tutorials 1, 4, 5 and 7 ideally completed first — this capstone composes their lessons.

## Learning aims
- Primary package aim: **compose** the whole chain yourself — your own DOE, run it, fit a
  surrogate on the resulting sweep, evaluate it at inputs you choose, couple it into a
  metamodel, and sample that metamodel both ways.
- Secondary aim: learn to distrust your own pipeline in the ways it deserves. Which design
  failures does a surrogate survive? Which are fatal? And which of the fatal ones does the
  reported uncertainty warn you about? (Fewer than you would hope.)

## Success criteria
You can, unaided:
- drive `validate → plan → run → surrogate fit → surrogate eval` on inputs you chose;
- say in one sentence what the `pymc_gp` backend actually fits, and where it will lie to you;
- write a `MetaModelSpec` that couples your surrogate to another variable, and say what
  `--method propagate` and `--method joint` each compute — and which one is inference;
- fill in the deliverable so another lab member can reproduce your run **from the digests**,
  not from your transcription.

## Why this tutorial matters

Up to now each tutorial has been one slice of the framework — validate / plan / run / fit /
eval / build / sample, each demonstrated on a fixed example. Here you compose them. Not to
produce something publishable in an hour, but so you can show another lab member you can
drive the whole framework end to end, on inputs you chose, without copy-paste.

**Coupling is behind you, not ahead of you.** T7 built a two-model coupling and T8 a
three-model chain — both on the pre-built *placeholder* surrogate artifacts that ship with
the repo. What you have never done is wire a coupling to a surrogate **you** fitted from
**your** sweep. Step 6 is that step, and it is the first place in the series where
`bayesmm meta sample --method joint` actually runs: T7's placeholder artifacts carry no
`backend_payload`, so joint sampling refuses on them by design, and T7 had to fall back to a
hand-built demo with a closed-form answer.

The framework's value is exactly this composability: every step is the same
spec → CLI → artifact contract, whether you are running a toy sweep that finishes in ten
seconds or coupling four biological models. The full-scale version of what you are about to
build lives in
`projects/tcr_signaling/` — see the closing pointer at the bottom.

## Pre-flight: what this capstone actually needs

Almost nothing. This notebook builds its own run store from scratch, so **no artifact from
an earlier tutorial is required.** The cell below checks only the files this notebook
actually opens, and raises if one is missing.

That is the whole point of the change. A status display you never branch on — one that
prints `MISSING (some steps may be skipped)` when no step can in fact be skipped — is
precisely the green light that means nothing, which this notebook's troubleshooting table
warns you about at the end. If a check cannot fail, it is decoration. Delete it or give it
teeth.

In [ ]:
# Cross-platform setup (Windows / macOS / Linux) — no shell, no PYTHONPATH prefix.
# Find the repo root so `src/` is importable, then load the shared tutorial helpers.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "src" / "bayesian_metamodeling").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from bayesian_metamodeling.tutorial import bootstrap, run_mm_cli, run_tool

root = bootstrap()  # chdir to repo root + ensure src/ on sys.path (idempotent)
ROOT = root
print("Repo root:", root)


# Preflight: detect PyMC. This notebook invokes `bayesmm surrogate fit/eval`
# with the pymc_gp backend, which requires `pymc` in THIS kernel's env. The
# main dev env (py314_bayesmm) deliberately ships without it — the backends
# live in their own envs — so detecting its absence here lets us skip the
# PyMC-specific steps with an actionable banner instead of a RuntimeError
# halfway down the notebook. Mirrors the SBI preflight in Tutorial 6.
import importlib.util as _ilu
import os as _os

PYMC_AVAILABLE = _ilu.find_spec("pymc") is not None

if not PYMC_AVAILABLE:
    _conda_env_name = _os.environ.get("CONDA_DEFAULT_ENV")
    _BANNER = "=" * 72
    print()
    print(_BANNER)
    print("  PREFLIGHT: PyMC backend missing — PyMC steps will be SKIPPED")
    print(_BANNER)
    print(f"  Kernel env path  : {sys.prefix}")
    if _conda_env_name:
        print(f"  Conda env        : {_conda_env_name}")
    print("  Missing package  : 'pymc' (pymc_gp surrogate backend)")
    print()
    print("  HOW TO FIX — use the dedicated backend env and re-run with that kernel:")
    print("    conda env create -f environment-pymc.yml")
    print("    conda activate py312_bayesmm_pymc")
    print("    python -m ipykernel install --user --name py312_bayesmm_pymc")
    print()
    print("  The rest of the notebook still runs; only the PyMC steps are skipped.")
    print(_BANNER)
    print()


In [ ]:
# What this capstone consumes: two checked-in spec templates and the toy program. Nothing
# else. `root` came from bootstrap() in the cell above — do NOT re-derive it here with a
# second, weaker cwd heuristic; that is how a notebook starts disagreeing with itself about
# where the repo is.
REQUIRED = {
    "model spec template": root / "tutorials/specs/model.toy.grid.json",
    "surrogate spec template": root / "tutorials/specs/surrogate.toy.pymc_gp.json",
    "toy program": root / "examples/toy_program/run.py",
}

print("Pre-flight — files this notebook opens:")
_missing = []
for _label, _path in REQUIRED.items():
    _ok = _path.exists()
    if not _ok:
        _missing.append(_label)
    print(f"  {'OK     ' if _ok else 'MISSING'}  {_label}: {_path.relative_to(root)}")

if _missing:
    raise FileNotFoundError(
        f"Capstone cannot start — missing: {', '.join(_missing)}. "
        "These ship with the repo; check you are on a complete checkout."
    )

print()
print("Nothing else is required. Earlier tutorials' run/surrogate/metamodel registries are")
print("irrelevant here: Step 1 creates its own store and clears it first.")

## Step 1: Pick your DOE

Edit `MY_GRID_A` and `MY_GRID_B` below — any values inside the input box `[0, 2]²`, any
lengths. The framework validates, plans and runs their Cartesian product. This is the muscle
T1 and T4 built; the loop takes about ten seconds. Change them and re-run — that is the point
of *your* DOE.

**Why the cell deletes the store before running.** A run store is *cumulative by design*.
`bayesmm run` writes each sweep into a fresh directory under `sweeps/` and replaces nothing —
which is exactly what lets `projects/tcr_signaling/` keep four models' sweeps side by side in
one store. But a surrogate fit reads **every** `sweep_rows.csv` under the store root it is
pointed at: `surrogates/dataset.py::_load_from_centralized_sweeps` does an `rglob` and
concatenates whatever carries its input columns.

So re-running this notebook with a new grid, without clearing, trains the next surrogate on
your new grid **plus every grid you tried before** — silently, with each repeated point
entering the likelihood again, which makes the posterior narrower than your data earns.
Nothing errors. Nothing warns.

Clearing the store turns "the fit used the points I just produced" from a hope into a fact —
and Step 2 prints the row count the loader actually hands to the fit, so you never take it on
faith. Both halves matter: the hygiene, and the check that the hygiene worked.

In [ ]:
import json
import shutil

# === EDIT ME ===
# Pick grid values for `a` and for `b`, inside the box [0, 2]². They are combined as the
# Cartesian product. Any lengths — nothing downstream hardcodes 5.
MY_GRID_A = [0.1, 0.6, 1.1, 1.6, 1.9]
MY_GRID_B = [0.1, 0.6, 1.1, 1.6, 1.9]
# ===============

CAPSTONE_STORE = "tmp/tutorials/capstone_store"
N_DOE = len(MY_GRID_A) * len(MY_GRID_B)

# Start from an empty store — see the note above. A run store accumulates by design, and a
# surrogate fit reads all of it.
shutil.rmtree(root / CAPSTONE_STORE, ignore_errors=True)

# Build a temporary spec by mutating the checked-in toy grid spec.
base_spec = json.loads((root / "tutorials/specs/model.toy.grid.json").read_text())
base_spec["model"]["name"] = "tutorial9_capstone_grid"
base_spec["design"]["grid"] = {"a": MY_GRID_A, "b": MY_GRID_B}
base_spec["storage"]["root"] = CAPSTONE_STORE

capstone_spec_path = root / "tmp/tutorials/specs/capstone.grid.json"
capstone_spec_path.parent.mkdir(parents=True, exist_ok=True)
capstone_spec_path.write_text(json.dumps(base_spec, indent=2, sort_keys=True))
CAPSTONE_SPEC_REL = str(capstone_spec_path.relative_to(root))

# Print the repo-RELATIVE path. An absolute path in a committed output tells the next reader
# where your home directory is and nothing they can use.
print(f"Wrote: {CAPSTONE_SPEC_REL}")
print(f"DOE: {len(MY_GRID_A)} a-values × {len(MY_GRID_B)} b-values = {N_DOE} points")
print(f"Store cleared and re-created at: {CAPSTONE_STORE}")

# Validate, plan, run.
run_mm_cli("validate", CAPSTONE_SPEC_REL)
run_mm_cli("plan", CAPSTONE_SPEC_REL)
run_mm_cli("run", CAPSTONE_SPEC_REL)

## Step 2: Fit a surrogate on your sweep

The cell below writes a surrogate spec whose `dataset_ref.run_store_root` points at the store
Step 1 just filled, checks how many rows the loader will actually hand to the fit, and fits.

### What `"backend": "pymc_gp"` actually fits — read this before you trust it

Despite the name, **there is no Gaussian process here.** `surrogates/backends.py` dispatches
`pymc_gp` to `_fit_pymc_bayesian_linear`, which builds

```
beta      ~ Normal(0, 2)        # one weight per input
intercept ~ Normal(0, 2)
sigma     ~ HalfNormal(1)
y         ~ Normal(intercept + x @ beta, sigma)
```

— Bayesian **linear regression**. No kernel, no covariance function, no `pm.gp` anywhere in
`src/`. The name is a misnomer, and it is worth knowing about because reasoning from the name
gets the behaviour backwards. A GP reverts toward its prior mean and widens its band as you
leave the training data, so it warns you. A linear model extrapolates its fitted plane
forever, and stays confident. You will measure both claims in Steps 3 and 4 rather than take
them from a paragraph.

### What a fit produces

Not a function — an **artifact**. `tmp/surrogate_artifacts/<id>/artifact.json` records the
backend, a digest of the spec, a digest of the training dataset, the seed, the versions of
python/pymc/numpy/scipy that produced it, and a pointer to the posterior draws of `beta`,
`intercept` and `sigma`. That artifact, not the object in this kernel's memory, is what T7's
couplings and Step 6 below consume **by reference**. It is also what makes the deliverable at
the end reproducible without you copying numbers.

In [ ]:
if not PYMC_AVAILABLE:
    print("Step 2 SKIPPED (PyMC missing in this kernel) — see preflight banner above.")
else:
    import json

    from bayesian_metamodeling.spec import SurrogateSpec
    from bayesian_metamodeling.storage.surrogate_store import find_latest_artifact_for_spec
    from bayesian_metamodeling.surrogates.dataset import load_tabular_dataset

    # Build a surrogate spec pointing at the capstone storage from Step 1.
    base_surr = json.loads((root / "tutorials/specs/surrogate.toy.pymc_gp.json").read_text())
    base_surr["name"] = "tutorial9_capstone_surrogate"
    base_surr["dataset_ref"]["run_store_root"] = CAPSTONE_STORE
    # `summary_config` picks WHICH element of the toy's output array `y` to fit.
    # index 0 is y[0] = a + b. The toy also writes y[1] = a·b — Step 4 fits that one.
    base_surr["summary_config"] = {"kind": "index", "index": 0}

    capstone_surr_path = root / "tmp/tutorials/specs/capstone.surrogate.pymc_gp.json"
    capstone_surr_path.write_text(json.dumps(base_surr, indent=2, sort_keys=True))
    CAPSTONE_SURR_REL = str(capstone_surr_path.relative_to(root))

    print(f"Wrote: {CAPSTONE_SURR_REL}")
    print(f"  dataset_ref.run_store_root: {CAPSTONE_STORE}")

    # Ask the loader what it will train on BEFORE fitting. This is the question Step 1's
    # rmtree exists to make answerable, and the answer is not "whatever I just ran" unless
    # someone checked.
    _x_train, _y_train, _dataset_digest = load_tabular_dataset(
        SurrogateSpec.model_validate(base_surr)
    )
    print(f"  rows the loader will hand to the fit: {_x_train.shape[0]}  (your DOE: {N_DOE})")
    assert _x_train.shape[0] == N_DOE, (
        f"The fit would train on {_x_train.shape[0]} rows, not your {N_DOE}. Something else "
        f"wrote into {CAPSTONE_STORE} — a surrogate fit reads every sweep under its store root."
    )

    run_mm_cli("surrogate", "fit", CAPSTONE_SURR_REL)

    # `bayesmm surrogate list` prints every artifact ever registered; here we want the one we
    # just made, which is also the id the deliverable and Step 6 need.
    CAPSTONE_ART_ID, CAPSTONE_ART_PATH = find_latest_artifact_for_spec(
        "tutorial9_capstone_surrogate"
    )
    print(f"\nYour surrogate artifact: {CAPSTONE_ART_ID}")
    print(f"  at: {CAPSTONE_ART_PATH}")

## Step 3: Evaluate at inputs you choose — and read what comes back

A trained surrogate predicts at *any* input, including ones you never ran the simulator at.
That is the point of having one.

But "predicts" is carrying a lot of weight in that sentence. Fill this in for yourself before
running anything — you will be asked for it again in the deliverable:

> A surrogate is ______ , fitted to ______ , which returns ______ rather than a number, and I
> should stop trusting it when ______ .

*(One defensible filling: a cheap probability model, fitted to a sweep's inputs and outputs,
which returns a distribution over the output rather than a number, and which I should stop
trusting where my design gave it no information about the direction I am asking about. Steps 4
and 6 are about that last blank; linear interpolation would satisfy the first three.)*

**Predict before you run.** Four query points, `--n 300`:

1. What will `sample_shape` be?
2. Will `summary.mean` change if you re-run with `--n 20`? Will `samples_preview`?
3. Your grid covers some sub-box of `[0, 2]²`. At `a = b = 10`, far outside it, what happens
   to `summary.std`? Does it grow enough to warn you?

Commit to three answers. The two cells after the next one check all three.

In [ ]:
if not PYMC_AVAILABLE:
    print("Step 3 SKIPPED (PyMC missing in this kernel) — see preflight banner above.")
else:
    import json

    # === EDIT ME ===
    # Pick 4 query points that are NOT in your DOE grid.
    QUERY_INPUTS = {
        "a": [0.35, 0.85, 1.45, 1.85],
        "b": [0.55, 1.25, 0.75, 1.15],
    }
    # ===============

    run_mm_cli(
        "surrogate", "eval", CAPSTONE_SURR_REL,
        "--inputs", json.dumps(QUERY_INPUTS),
        "--n", "300",
    )

### Every field in that block, in one table

| field | what it is |
|---|---|
| `artifact_id` | which fitted artifact answered — the same id you put in a `MetaModelSpec`'s `surrogate_refs` |
| `n` (top level) | how many posterior-predictive draws were **returned to you** |
| `sample_shape` | `[query points, n, outputs]` — one draw array per query point |
| `summary.mean`, `summary.std` | computed from the **fit's own posterior draws**, analytically — `backends.py::summary()` takes no `n` at all |
| `summary.n` | a *different* `n`: the number of query rows. Two fields, one letter, two meanings |
| `summary.posterior_draws` | how many MCMC draws the fit retained — your spec's `backend_config.draws` |
| `summary.output_correlation` | `diagonal` = outputs modelled independently. With ≥2 outputs, `full` fits a covariance across them instead |
| `samples_preview` | the first few draws, so you can see the array is real |

The consequence worth internalising: **`--n` sizes the sample array you get back and nothing
else.** It cannot move `mean` or `std`, because those are not Monte-Carlo estimates from those
draws — `eval_surrogate` calls `model.sample(inputs, n, seed)` and then, separately,
`model.summary(inputs)`. The next cell demonstrates that instead of asking you to believe it,
and then walks the surrogate far outside its training box.

In [ ]:
if not PYMC_AVAILABLE:
    print("Step 3 probes SKIPPED (PyMC missing in this kernel) — see preflight banner above.")
else:
    import json

    import numpy as np

    from bayesian_metamodeling.spec import SurrogateSpec
    from bayesian_metamodeling.surrogates import eval_surrogate

    capstone_spec = SurrogateSpec.model_validate(json.loads(capstone_surr_path.read_text()))

    # --- Prediction 1 and 2: does --n move anything in `summary`? ---
    _r300 = eval_surrogate(spec=capstone_spec, inputs_payload=QUERY_INPUTS, n=300)
    _r20 = eval_surrogate(spec=capstone_spec, inputs_payload=QUERY_INPUTS, n=20)
    print("sample_shape  at n=300 :", _r300["sample_shape"])
    print("sample_shape  at n=20  :", _r20["sample_shape"])
    _dmean = float(np.max(np.abs(np.asarray(_r300["summary"]["mean"], dtype=float)
                                 - np.asarray(_r20["summary"]["mean"], dtype=float))))
    _dstd = float(np.max(np.abs(np.asarray(_r300["summary"]["std"], dtype=float)
                                - np.asarray(_r20["summary"]["std"], dtype=float))))
    print(f"largest change in summary.mean between n=300 and n=20: {_dmean:.3e}")
    print(f"largest change in summary.std  between n=300 and n=20: {_dstd:.3e}")
    print("summary.posterior_draws:", _r300["summary"]["posterior_draws"],
          "(= backend_config.draws in your spec, not --n)")
    print("summary.n              :", _r300["summary"]["n"], "(= number of query rows)")
    print("summary.output_correlation:", _r300["summary"]["output_correlation"])

    # --- Prediction 3: walk out of the box and watch the band ---
    FAR_INPUTS = {"a": [1.0, 5.0, 10.0, 20.0], "b": [1.0, 5.0, 10.0, 20.0]}
    _far = eval_surrogate(spec=capstone_spec, inputs_payload=FAR_INPUTS, n=50)
    _fm = np.asarray(_far["summary"]["mean"], dtype=float)
    _fs = np.asarray(_far["summary"]["std"], dtype=float)
    _ft = np.asarray(FAR_INPUTS["a"]) + np.asarray(FAR_INPUTS["b"])  # truth: y[0] = a + b

    print("\nLeaving the training box (truth here is y = a + b):")
    print(f"  {'a=b':>6} {'predicted':>12} {'truth':>10} {'|error|':>11} {'reported std':>14}")
    for _a, _m, _t, _s in zip(FAR_INPUTS["a"], _fm, _ft, _fs):
        print(f"  {_a:6.1f} {_m:12.5f} {_t:10.3f} {abs(_m - _t):11.2e} {_s:14.2e}")

    IN_BOX_STD = float(np.mean(np.asarray(_r300["summary"]["std"], dtype=float)))
    print(f"\n  in-box reported std (mean over your 4 query points): {IN_BOX_STD:.2e}")
    print(f"  reported std at a=b=20, ten times outside the box  : {_fs[-1]:.2e}")
    print(f"  ratio: {_fs[-1] / IN_BOX_STD:.1f}×  — and both are still ~1e-6.")
    print("\n  The mean is exactly right out there, because the truth y = a + b lives inside")
    print("  the family this backend fits. The band grew a little (the weight posterior has a")
    print("  little spread, and it is multiplied by a larger x) but it never says 'I don't")
    print("  know'. There is no training hull here and no reversion to a prior mean: those are")
    print("  Gaussian-process behaviours, and this backend is not a Gaussian process.")

## Step 4: Break it on purpose — three ways, and which ones the band warns you about

Step 3 looked like a triumph: near-exact predictions, microscopic error bars, and exact
extrapolation ten times outside the box. Every one of those results follows from a single
coincidence — **the truth you fitted, `y[0] = a + b`, is exactly linear, and `pymc_gp` fits
exactly linear models.** The family contains the truth, so nothing else can go wrong.

Real models are not in your surrogate's family, and real designs miss things. The cell below
runs three variants and prints, for each, the error at *your* query points next to the width
the surrogate reported there. The column to read is **error / width**: how many of the
surrogate's own "standard deviations" it was wrong by. A width that means something keeps that
ratio at order 1 or below; a ratio in the thousands means the width is fiction.

1. **Tiny coverage** — same truth, but a DOE crammed into a small corner of the box.
2. **A direction never excited** — a legal DOE of the same size in which `a` never varies.
3. **Wrong family** — your DOE exactly, fitted to the toy's *other* output, `y[1] = a·b`,
   which is not linear.

**Predict before you run.** Which of the three does the surrogate survive? Of the ones it
fails, which failures would its own error bars have warned you about? Write down your ranking
before the numbers appear; it is easy to agree with a result you have already read.

In [ ]:
if not PYMC_AVAILABLE:
    print("Step 4 SKIPPED (PyMC missing in this kernel) — see preflight banner above.")
else:
    import json
    import shutil

    import numpy as np

    from bayesian_metamodeling.spec import SurrogateSpec
    from bayesian_metamodeling.storage.surrogate_store import find_latest_artifact_for_spec
    from bayesian_metamodeling.surrogates import eval_surrogate

    def sweep_variant(store, model_name, grid_a, grid_b):
        """Clear a store and run one DOE into it. Same three CLI verbs as Step 1."""
        shutil.rmtree(root / store, ignore_errors=True)
        spec = json.loads((root / "tutorials/specs/model.toy.grid.json").read_text())
        spec["model"]["name"] = model_name
        spec["design"]["grid"] = {"a": list(grid_a), "b": list(grid_b)}
        spec["storage"]["root"] = store
        path = root / f"tmp/tutorials/specs/{model_name}.json"
        path.write_text(json.dumps(spec, indent=2, sort_keys=True))
        run_mm_cli("run", str(path.relative_to(root)))

    def fit_variant(store, surrogate_name, index):
        """Fit pymc_gp on `store`, selecting element `index` of the toy's output array."""
        spec = json.loads((root / "tutorials/specs/surrogate.toy.pymc_gp.json").read_text())
        spec["name"] = surrogate_name
        spec["dataset_ref"]["run_store_root"] = store
        spec["summary_config"] = {"kind": "index", "index": index}
        path = root / f"tmp/tutorials/specs/{surrogate_name}.json"
        path.write_text(json.dumps(spec, indent=2, sort_keys=True))
        run_mm_cli("surrogate", "fit", str(path.relative_to(root)))
        return SurrogateSpec.model_validate(spec)

    def score(spec, inputs, truth):
        """Error vs truth, and the width the surrogate reported at the same points."""
        result = eval_surrogate(spec=spec, inputs_payload=inputs, n=50)
        mean = np.asarray(result["summary"]["mean"], dtype=float)
        std = np.asarray(result["summary"]["std"], dtype=float)
        err = np.abs(mean - np.asarray(truth, dtype=float))
        return {
            "mean": mean, "std": std, "err": err,
            "mae": float(err.mean()),
            "width": float(std.mean()),
            "ratio": float((err / std).mean()),
        }

    qa = np.asarray(QUERY_INPUTS["a"], dtype=float)
    qb = np.asarray(QUERY_INPUTS["b"], dtype=float)
    TRUTH_SUM = qa + qb
    TRUTH_PROD = qa * qb

    BREAKAGE = {}
    BREAKAGE["your DOE, y = a+b"] = score(capstone_spec, QUERY_INPUTS, TRUTH_SUM)

    # (1) tiny coverage: the same number of points, crammed into the corner [0, 0.08]².
    corner = [round(v, 4) for v in np.linspace(0.0, 0.08, len(MY_GRID_A))]
    CORNER_AREA_PCT = 100.0 * (0.08 / 2.0) ** 2
    print(f"corner DOE spans {corner[0]} to {corner[-1]} on each axis "
          f"= {CORNER_AREA_PCT:.2f}% of the [0, 2]² box's area")
    sweep_variant("tmp/tutorials/capstone_store_corner", "tutorial9_corner_grid", corner, corner)
    corner_spec = fit_variant("tmp/tutorials/capstone_store_corner", "tutorial9_corner_doe", 0)
    BREAKAGE["tiny corner DOE, y = a+b"] = score(corner_spec, QUERY_INPUTS, TRUTH_SUM)

    # (2) a direction the design never excites: `a` is held fixed, `b` spans your range.
    flat_a = [1.0] * len(MY_GRID_A)
    sweep_variant("tmp/tutorials/capstone_store_flat", "tutorial9_flat_grid", flat_a, MY_GRID_B)
    flat_spec = fit_variant("tmp/tutorials/capstone_store_flat", "tutorial9_flat_doe", 0)
    BREAKAGE["a never varies, y = a+b"] = score(flat_spec, QUERY_INPUTS, TRUTH_SUM)

    # (3) wrong family: your data, your DOE, but the toy's nonlinear output y[1] = a·b.
    wrong_spec = fit_variant(CAPSTONE_STORE, "tutorial9_wrong_family", 1)
    BREAKAGE["your DOE, y = a·b"] = score(wrong_spec, QUERY_INPUTS, TRUTH_PROD)
    WRONG_ART_ID, WRONG_ART_PATH = find_latest_artifact_for_spec("tutorial9_wrong_family")

    print("\n=== Four fits, scored at YOUR query points ===")
    print(f"{'variant':<28} {'MAE':>11} {'reported width':>16} {'error / width':>15}")
    for label, s in BREAKAGE.items():
        print(f"{label:<28} {s['mae']:11.3e} {s['width']:16.3e} {s['ratio']:15.3g}")

    # The wrong-family fit, pushed outside the box where a plane and a product diverge.
    FAR2 = {"a": [5.0, 10.0], "b": [5.0, 10.0]}
    far = eval_surrogate(spec=wrong_spec, inputs_payload=FAR2, n=50)
    FAR_MEAN = np.asarray(far["summary"]["mean"], dtype=float)
    FAR_STD = np.asarray(far["summary"]["std"], dtype=float)
    FAR_TRUTH = np.asarray(FAR2["a"]) * np.asarray(FAR2["b"])
    FAR_RATIO = float(np.mean(np.abs(FAR_MEAN - FAR_TRUTH) / FAR_STD))
    print("\n=== The wrong-family fit, extrapolated (truth y = a·b) ===")
    print(f"  {'a=b':>5} {'predicted':>12} {'truth':>10} {'|error|':>10} {'width':>9} {'err/width':>11}")
    for a_v, m, t, s in zip(FAR2["a"], FAR_MEAN, FAR_TRUTH, FAR_STD):
        print(f"  {a_v:5.1f} {m:12.3f} {t:10.1f} {abs(m - t):10.2f} {s:9.2f} {abs(m - t) / s:11.1f}")

### What those four rows teach

Read the **error / width** column; the rest is context.

- **Coverage is not the constraint.** The corner DOE saw the percentage of the box the cell
  printed — a sliver — and still predicts across the whole box about as well as your grid did.
  For a *correctly specified* model, where you sampled barely matters. What matters is whether
  the design **identifies the parameters**.

- **Rank is the constraint, and the band does not report it.** The DOE in which `a` never
  varies is the same size and spans the same `b` range, and its row is the only one whose
  error/width is not of order 1 — by a factor the cell prints in full. Its MAE is orders of
  magnitude worse than the other three, while the width it reports stayed down at the same
  microscopic scale as the fits that were right.
  Nothing in that data distinguishes "y grows with a" from "y ignores a": the weight on `a` is
  unidentified, so the fitted chain never moved along that direction, and the spread it
  reports along it is a fact about the chain, not about the truth. **A number can be a green
  light that means nothing happened** — the same failure shape as a test suite that passes
  because every test skipped.

- **A wrong family is unremarkable in-box and lethal outside it.** Fitted to `y = a·b`, the
  linear surrogate is off inside the training box by a fraction of the width it reports:
  honest, boring, exactly what a band is for. Now read the extrapolation block, and read the
  err/width column in particular: out there the surrogate is wrong by *tens of its own standard
  deviations*. The band covers **parameter** uncertainty inside the assumed family. It never
  covers the family being the wrong shape.

So the sentence T5 asks you to carry forward — *the surrogate's uncertainty is what makes it
safe to use in place of the model* — needs a rider you should be able to state from memory:
**only within the family you assumed, and only along directions your design actually varied.**

And one claim to correct if you have met it phrased loosely: it is *not* that "no fit recovers
structure the design never sampled". The corner DOE recovered the structure of the whole box
from a corner of it. It is that no fit recovers a **direction the design never varied** — and
it will not tell you that it failed.

## Step 5: Visualize what your two surrogates believe

Two panels, same machinery, same training rows — the only difference is which column of the
sweep was fitted.

**Left**, `y[0] = a + b`: the family contains the truth. The predictions sit on the line, and
**the error bars are so small they are invisible** — that is not a plotting bug, it is the fit
being exact. The annotation next to each point carries the actual width; read that, not the
marker.

**Right**, `y[1] = a·b`: the family does not contain the truth. Here the bars are visible, and
the points sit off the curve by a fraction of a bar. This is what a *normal* surrogate fit
looks like, and it is the picture to keep in your head as the default — the left panel is the
special case, not this one.

In [ ]:
if not PYMC_AVAILABLE:
    print("Step 5 (plot) SKIPPED (PyMC missing in this kernel) — see preflight banner above.")
else:
    import csv

    import matplotlib.pyplot as plt
    import numpy as np

    # Read the training rows the SAME way the fit does: every sweep_rows.csv under the store
    # root, not an arbitrary one picked by an unordered glob. After Step 1's rmtree there is
    # exactly one — and the assert below is what makes that a statement instead of an
    # assumption.
    train_a, train_b, train_y0, train_y1 = [], [], [], []
    for csv_path in sorted((root / CAPSTONE_STORE / "sweeps").rglob("sweep_rows.csv")):
        with open(csv_path, newline="", encoding="utf-8") as handle:
            for row in csv.DictReader(handle):
                if row.get("status") != "success":
                    continue
                train_a.append(float(row["a"]))
                train_b.append(float(row["b"]))
                train_y0.append(float(row["y__0"]))
                train_y1.append(float(row["y__1"]))
    train_a = np.asarray(train_a)
    train_b = np.asarray(train_b)
    assert len(train_a) == N_DOE, (
        f"Plot is reading {len(train_a)} training rows but your DOE has {N_DOE}."
    )

    query_sum = np.asarray(QUERY_INPUTS["a"]) + np.asarray(QUERY_INPUTS["b"])
    query_prod = np.asarray(QUERY_INPUTS["a"]) * np.asarray(QUERY_INPUTS["b"])

    # Each panel plots y against its own truth expression, so the analytical truth is the
    # 45° line in both — the deviation from it is the whole point of the figure.
    panels = [
        ("y[0] = a + b  —  family contains the truth", "a + b",
         train_a + train_b, np.asarray(train_y0), query_sum,
         BREAKAGE["your DOE, y = a+b"]),
        ("y[1] = a · b  —  family does NOT contain the truth", "a · b",
         train_a * train_b, np.asarray(train_y1), query_prod,
         BREAKAGE["your DOE, y = a·b"]),
    ]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, (title, xlabel, tx, ty, qx, res) in zip(axes, panels):
        grid = np.linspace(0, max(tx.max(), qx.max()) * 1.05, 100)
        ax.plot(grid, grid, "k:", alpha=0.6, label="analytical truth")
        ax.scatter(tx, ty, marker="x", s=60, c="tab:gray", alpha=0.7,
                   label=f"YOUR training rows (N={len(tx)})", zorder=3)
        ax.errorbar(qx, res["mean"], yerr=res["std"], fmt="o", color="tab:blue",
                    markersize=9, capsize=5, zorder=4,
                    label="surrogate posterior predictive (mean ± std)")
        for x_v, m, s in zip(qx, res["mean"], res["std"]):
            ax.annotate(f"({m:.2f} ± {s:.0e})", xy=(x_v, m), xytext=(5, 6),
                        textcoords="offset points", fontsize=8, alpha=0.75)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("y")
        ax.grid(True, alpha=0.3)
        ax.legend(loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()

    print("=== Capstone numeric summary ===")
    print(f"Training rows          : {len(train_a)}")
    print(f"Query points           : {len(query_sum)}")
    for label in ("your DOE, y = a+b", "your DOE, y = a·b"):
        res = BREAKAGE[label]
        print(f"\n{label}")
        print(f"  predicted means      : {[round(float(v), 4) for v in res['mean']]}")
        truth = query_sum if label.endswith("a+b") else query_prod
        print(f"  analytical truth     : {[round(float(v), 4) for v in truth]}")
        print(f"  predictive std       : {[f'{float(v):.2e}' for v in res['std']]}")
        print(f"  mean absolute error  : {res['mae']:.3e}")
        print(f"  error / width        : {res['ratio']:.3g}")

## Step 6: Wire your surrogate into a metamodel

Everything so far has been *one* model. A **metamodel** is what you write when you have more
than one — or one model and a measurement — and you need them to agree.

A `MetaModelSpec` is nothing but three lists, plus references to fitted surrogates:

- **`variables`** — the named quantities in play. Yours: `a`, `b` (the inputs), `y` (the
  surrogate's output), and `y_obs`, an experimental measurement of the same physical quantity.
- **`priors`** — what you believed about each one before this analysis.
- **`couplings`** — the assertions that make it a *meta*model. A `gaussian_link` from `y` to
  `y_obs` with `sigma = 0.1` asserts: *these two are the same quantity, and I will let them
  disagree by about 0.1.* (T7's vocabulary: `deterministic` is the hard version —
  `target = transform(source)` exactly, enforced with a penalty.)
- **`surrogate_refs`** — a path or artifact id per fitted surrogate. The IR builder turns each
  into a **likelihood factor**: `p(y | a, b)`, as your fit believes it.

### Which surrogate — and why the answer is the "wrong-family" one

Step 4's `a·b` fit is the *realistic* one. Look back at the width column for your `a + b`
surrogate in Steps 3 and 4: it asserts `y = a + b` to something like seven decimal places.
As a likelihood factor that is not a soft constraint, it is a knife edge. We build the
metamodel **both ways** below so you can watch what that does to a sampler — compare the two
blocks' `accept_rate`, effective sample sizes and `poorly mixed` lines. A surrogate with
dishonestly small residual noise does not merely mislead you downstream; it makes the
downstream uninferable.

### `propagate` vs `joint`, in the terms of this spec

| | `--method propagate` (default) | `--method joint` |
|---|---|---|
| what it does | draw every variable from its prior, then overwrite each coupling target with `transform(source)` (+ noise) | Metropolis on the full joint log-density |
| your surrogate | **never called** — the compiled model is invoked with `surrogates={}` | conditioned on |
| the coupling's source (`y`) | untouched, stays at its prior | tightened |
| the coupling target's own prior | discarded — it is overwritten | respected, as one factor among several |
| honest name for the output | forward uncertainty propagation | a posterior |

**Predict before you run.** Under `propagate`: what will `sd(a)` be, given `a ~ Normal(1, 0.5)`
and no coupling touching `a`? And `y_obs` has both a prior of `Normal(2.6, 0.1)` and is the
target of the coupling — which of the two wins? Then under `joint`: does `sd(a)` go up, down,
or stay put, and *why would information reach `a` at all*?

In [ ]:
if not PYMC_AVAILABLE:
    print("Step 6 SKIPPED (PyMC missing in this kernel) — see preflight banner above.")
else:
    import json

    import numpy as np

    # === EDIT ME ===
    # The scientific setup: you believe a and b are near 1, you are vague about y, and an
    # experiment measured the same quantity y at 2.6 with a standard error of 0.1.
    PRIOR_A = (1.0, 0.5)
    PRIOR_B = (1.0, 0.5)
    PRIOR_Y = (2.0, 2.0)      # deliberately vague — let the surrogate and the data speak
    MEASURED_Y = (2.6, 0.1)   # the experiment's value and its standard error
    LINK_SIGMA = 0.1          # how much model and measurement are allowed to disagree
    # ===============

    def write_meta_spec(name, artifact_path):
        spec = {
            "schema_version": "1.0",
            "name": name,
            "ppl_backend": "pymc",
            "surrogate_refs": [str(artifact_path)],
            "variables": [{"name": v, "type": "scalar"} for v in ("a", "b", "y", "y_obs")],
            "priors": [
                {"variable": "a", "distribution": {"kind": "normal", "loc": PRIOR_A[0], "scale": PRIOR_A[1]}},
                {"variable": "b", "distribution": {"kind": "normal", "loc": PRIOR_B[0], "scale": PRIOR_B[1]}},
                {"variable": "y", "distribution": {"kind": "normal", "loc": PRIOR_Y[0], "scale": PRIOR_Y[1]}},
                {"variable": "y_obs", "distribution": {"kind": "normal", "loc": MEASURED_Y[0], "scale": MEASURED_Y[1]}},
            ],
            "couplings": [
                {"kind": "gaussian_link", "source": "y", "target": "y_obs",
                 "transform": {"kind": "identity"}, "sigma": LINK_SIGMA},
            ],
        }
        path = root / f"tmp/tutorials/specs/{name}.json"
        path.write_text(json.dumps(spec, indent=2, sort_keys=True))
        return str(path.relative_to(root))

    def latest_samples(meta_name, method):
        """Fetch the newest stored sample set for (spec name, method).

        Note the two sampling paths label the same field differently — `sample_metamodel`
        writes `ir_name`, `sample_joint_to_store` writes `name` — so accept either. `method`
        is the field that actually tells you what the numbers mean.
        """
        registry = json.loads((root / "tmp/metamodel_samples_registry.json").read_text())
        rows = [e for e in registry.values()
                if (e.get("name") or e.get("ir_name")) == meta_name and e.get("method") == method]
        if not rows:
            raise RuntimeError(f"No stored samples for {meta_name} with method={method}.")
        entry = max(rows, key=lambda e: e.get("created_at", ""))
        variables = json.loads((root / entry["samples_dataset_path"]).read_text())["variables"]
        info = json.loads((root / entry["inference_data_path"]).read_text())
        return {k: np.asarray(v, dtype=float).reshape(-1) for k, v in variables.items()}, info

    META_RESULTS = {}
    for label, meta_name, artifact in (
        ("wrong-family surrogate (y = a·b)", "tutorial9_capstone_meta", WRONG_ART_PATH),
        ("exact surrogate (y = a+b)", "tutorial9_capstone_meta_exact", CAPSTONE_ART_PATH),
    ):
        print(f"\n{'=' * 78}\n{label}\n{'=' * 78}")
        rel = write_meta_spec(meta_name, artifact)
        run_mm_cli("meta", "build", rel)
        run_mm_cli("meta", "sample", rel, "--draws", "2000", "--tune", "500",
                   "--chains", "2", "--seed", "9")
        run_mm_cli("meta", "sample", rel, "--draws", "2000", "--tune", "500",
                   "--chains", "2", "--seed", "9", "--method", "joint")
        prop, _ = latest_samples(meta_name, "prior_propagation")
        joint, joint_info = latest_samples(meta_name, "random_walk_metropolis")
        META_RESULTS[meta_name] = {"label": label, "prop": prop, "joint": joint,
                                   "info": joint_info}

    PRIORS = {"a": PRIOR_A, "b": PRIOR_B, "y": PRIOR_Y, "y_obs": MEASURED_Y}
    for meta_name, res in META_RESULTS.items():
        print(f"\n=== {res['label']} — {meta_name} ===")
        print(f"{'variable':<9} {'prior mean':>11} {'prior sd':>9} | "
              f"{'prop mean':>10} {'prop sd':>9} | {'joint mean':>11} {'joint sd':>9}")
        for name, (loc, scale) in PRIORS.items():
            p, j = res["prop"][name], res["joint"][name]
            print(f"{name:<9} {loc:11.3f} {scale:9.3f} | {p.mean():10.3f} {p.std(ddof=1):9.3f} "
                  f"| {j.mean():11.3f} {j.std(ddof=1):9.3f}")
        info = res["info"]
        stuck = info.get("poorly_mixed") or []
        ess = info.get("ess", {})
        print(f"  joint accept_rate: {info['accept_rate']:.3f}   "
              f"draws retained: {info['draws'] * info['chains']}")
        print(f"  effective sample size: "
              + ", ".join(f"{k}={v:.0f}" for k, v in sorted(ess.items())))
        print(f"  poorly mixed: {stuck if stuck else 'none'}")

### Reading those columns

**`propagate` reproduced your priors, and that is not a bug — it is the definition.** `a` and
`b` came back at their prior mean and prior sd because nothing in `propagate` conditions on
anything. It draws every variable from its prior and then overwrites each coupling target. Your
surrogate was never called (`sample_metamodel` invokes the compiled model with
`surrogates={}`), so the fit you spent Steps 1-4 building contributed **exactly nothing** to
those numbers. This is a perfectly good answer to *"what does my coupling imply downstream if I
take my priors at face value?"*. It is not inference, and calling its output a posterior is
wrong.

**Look at the `y_obs` row.** Its prior said sd `0.1`. Under `propagate` it came back with the
width of `y`'s prior instead, because the coupling *overwrote* it — propagate discards the
prior on any coupling target. If you ever need one row to explain why the default method is
not inference, it is that one.

**Under `joint`, both ends of the coupling move.** `y` is pulled toward the measurement, and —
this is the whole point of the curriculum — `a` and `b` moved too. Their mean shifted away from
the prior and their sd shrank. Nothing in the spec connects the measurement to `a` directly;
the information travelled *backwards through the surrogate likelihood*, from a measurement of
an output to the inputs that produce it. That is what `propagate` cannot do by construction,
and it is why `--method joint` exists.

**Now compare the two blocks' diagnostics** — `accept_rate`, the per-variable effective sample
sizes, and the `poorly mixed` line, plus any warning the CLI printed after the joint run. The
metamodel built on the exact `a + b` surrogate is the one in trouble. Its surrogate pins
`y = a + b` to the last decimal, so the posterior lives on a razor-thin ridge; tuning shrinks
every proposal to match it, and a coordinate-wise random walk then accepts a stream of
microscopic moves while exploring almost nothing. Its `sd` column may *look* like a sharper
answer. **It is not a tighter answer; it is not an answer** — an effective sample size in the
single digits out of thousands of retained draws is a chain reporting on itself. The lesson
generalises well past this notebook: a surrogate that under-reports its own residual noise
poisons every inference built on top of it, and the symptom shows up as a sampler diagnostic
rather than as an error.

Why Metropolis at all, rather than NUTS? A fitted surrogate's `log_prob` is an opaque Python
callable with no gradient, so a gradient-free sampler is what this model admits. That costs
efficiency, not correctness. Expressing every backend's `log_prob` as a PyTensor graph is what
would unlock NUTS, and it is the natural next piece of work.

### When this whole approach is the wrong tool

Worth being able to say out loud, because the framework will never refuse:

- **When you can afford to run the model.** A surrogate is a lossy compression of a simulator.
  If the simulator is cheap, sample the simulator. You accept surrogate error to buy speed; if
  you are not buying speed, you are only paying error.
- **When you cannot check the family.** You could audit `pymc_gp` here because the toy's truth
  was known. On a real model you usually cannot, so the honest move is held-out points the fit
  never saw — and Step 4 is the reason a plain "the band looks small" is not a substitute.
- **When the coupling is what is in doubt.** A metamodel takes your couplings as *given*. If
  "these two quantities are really the same" is the hypothesis under test, this machinery will
  assume your hypothesis and hand you a confident posterior conditional on it.
- **When the joint is a knife edge.** You have now seen a chain fail on a surrogate that was
  too confident. Near-deterministic couplings and near-noiseless surrogates make geometry no
  gradient-free sampler can walk. Loosen the coupling, refit with honest noise, or use a
  method with gradients.
- **When you need a decision, not a distribution.** Everything here produces a posterior. Turning
  one into an action needs a loss function, and the framework does not have one.

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `PREFLIGHT: PyMC backend missing` | Steps 2-6 need the `pymc_gp` backend | Use the `py312_bayesmm_pymc` or `py312_bayesmm_all` kernel. The steps skip cleanly otherwise — but then the capstone has demonstrated nothing. |
| The fit trained on more rows than your DOE | A run store is cumulative; a fit reads every `sweep_rows.csv` under its root | Clear the store before the sweep, as Step 1 does, and check the count Step 2 prints. Never infer the training size from what you *ran*. |
| `dataset_ref` points at an empty store | Step 1's sweep wrote somewhere else | `run_store_root` in the surrogate spec must match the sweep's `storage.root` exactly. Mismatched paths are the most common capstone failure. |
| Surrogate trains but predicts nonsense | Input names in the surrogate spec don't match the sweep columns | Open `sweep_rows.csv` and compare its headers against `inputs`. |
| Predictions are wrong but the band is tiny | Your design never varied one input, or the truth is outside the fitted family | Step 4 is this failure, twice. The band is not a lie detector; a held-out point is. |
| `Joint sampling unavailable: ... has no 'backend_payload'` | The referenced artifact is a placeholder, not a fit | Only real fits can be conditioned on. `--method propagate` never notices the difference, which is the point. |
| `WARNING: N variable(s) barely moved` after `--method joint` | A surrogate or coupling is far sharper than the priors, so the posterior sits on a ridge | Loosen the coupling `sigma`, or refit the surrogate on data with honest noise. Do **not** read the summaries — they are a report on a stuck chain. |
| Self-check passes but you learned nothing | Every step skipped on preflight | A green self-check that says "skipped per preflight" means the pipeline never ran. Read the beacon text, not just its presence. |

**Take those last two rows together.** They are the failure mode this whole curriculum keeps
circling: a status that reports success because nothing happened — a suite that passes because
every test skipped, a chain that reports a tight posterior because it never moved, a preflight
that prints FOUND for things nothing depends on. Whenever you build your own pipeline, ask what
its green light would look like if every step silently did nothing, and make sure that state is
distinguishable from success.

## Final deliverable

Two halves, and the split is the lesson.

**The machine's half** is the cell below: it assembles the report from the artifacts
themselves — the spec digest, the dataset digest, the seed, the dependency versions, the
metamodel sample ids. Nothing is transcribed, so nothing can be transcribed wrong. *Another lab
member reproduces your run from the digest chain and the seed, not from numbers you typed into
a markdown cell* — the digests are what make the claim checkable at all.

**Your half** is below that: three sentences no artifact can write for you. Edit them in place.

---

### Capstone interpretation (edit this)

**Date**: `<YYYY-MM-DD>` **Author**: `<your name>`

1. *A surrogate is ______ , fitted to ______ , which returns ______ rather than a number, and I
   should stop trusting it when ______ .* (Step 3 asked you to fill this in; write your final
   version here.)

2. *My coupling asserts ______ . Under `propagate` the sd of `a` was ______ and under `joint`
   it was ______ , and the reason it changed is ______ .*

3. *If I had to redo this on a model whose truth I could not check, the one thing I would change
   about my DOE is ______ , because ______ .*

---

If another lab member can read the printed report, run this notebook, and land on the same
digests, **you passed.** The point isn't perfection; it's that your claim is checkable by
someone who does not trust you.

In [ ]:
if not PYMC_AVAILABLE:
    print("Report SKIPPED (PyMC missing in this kernel) — see preflight banner above.")
else:
    import json

    import numpy as np

    art = json.loads((root / CAPSTONE_ART_PATH).read_text())
    wrong_art = json.loads((root / WRONG_ART_PATH).read_text())
    meta = META_RESULTS["tutorial9_capstone_meta"]

    print("### Capstone report (generated — do not hand-edit)")
    print()
    print("**Design**")
    print(f"- MY_GRID_A = {MY_GRID_A}")
    print(f"- MY_GRID_B = {MY_GRID_B}")
    print(f"- training rows: {N_DOE}   store: {CAPSTONE_STORE}")
    print(f"- model spec: {CAPSTONE_SPEC_REL}")
    print()
    print("**Surrogate (provenance straight out of artifact.json)**")
    for label, payload in (("y[0] = a+b", art), ("y[1] = a·b", wrong_art)):
        print(f"- {label}: artifact_id={payload['artifact_id']}")
        print(f"    backend={payload['backend']}  seed={payload['seed']}")
        print(f"    spec_digest={payload['spec_digest'][:16]}…  "
              f"dataset_digest={payload['dataset_digest'][:16]}…")
        print(f"    created_at={payload['created_at']}")
    print(f"- dependency_versions: "
          + ", ".join(f"{k}={v}" for k, v in sorted(art["dependency_versions"].items())
                      if k in {"python", "pymc", "numpy", "scipy"}))
    print()
    print("**Prediction at your query points**")
    print(f"- query a: {QUERY_INPUTS['a']}")
    print(f"- query b: {QUERY_INPUTS['b']}")
    for label in ("your DOE, y = a+b", "your DOE, y = a·b"):
        res = BREAKAGE[label]
        print(f"- {label}: MAE={res['mae']:.3e}  mean width={res['width']:.3e}  "
              f"error/width={res['ratio']:.3g}")
    print(f"- rank-deficient DOE (a never varies): "
          f"MAE={BREAKAGE['a never varies, y = a+b']['mae']:.3e}  "
          f"width={BREAKAGE['a never varies, y = a+b']['width']:.3e}")
    print()
    print("**Metamodel (wrong-family surrogate, the well-conditioned one)**")
    print(f"- priors: a~N{PRIOR_A}  b~N{PRIOR_B}  y~N{PRIOR_Y}  y_obs~N{MEASURED_Y}")
    print(f"- coupling: gaussian_link y → y_obs, sigma={LINK_SIGMA}")
    for name in ("a", "b", "y", "y_obs"):
        print(f"- sd({name}): prior={PRIORS[name][1]:.3f}  "
              f"propagate={meta['prop'][name].std(ddof=1):.3f}  "
              f"joint={meta['joint'][name].std(ddof=1):.3f}")
    print(f"- joint accept_rate={meta['info']['accept_rate']:.3f}  "
          f"poorly_mixed={meta['info'].get('poorly_mixed') or 'none'}")

## Recap: the whole pipeline in one view

You have now built, unaided, the chain the curriculum opened with — all of it, including the
last arrow:

```
Spec  →  DOE plan  →  Sweep  →  sweep_rows.csv  →  Surrogate  →  Metamodel
 T3       T4          T1/T2      T1               T5/T6          T7/T8/T9
```

Five ideas worth keeping:

1. **The spec is the contract.** Toy and published models are the same shape — that was T2's
   real lesson, not the biology. The same verbs drove your toy sweep here and drive four
   biological models in `projects/tcr_signaling/`.
2. **Rank, not coverage, is what a design has to deliver.** A DOE confined to a corner of the
   box recovered the whole box. A DOE that never varied one input was catastrophically wrong at
   the same size. Ask of any design: *which direction does this fail to excite?*
3. **Uncertainty is the deliverable — with a rider.** A point estimate from a surrogate is a
   guess; the band is what makes it evidence. But the band covers parameter uncertainty *inside
   the family you assumed and along directions your design varied*. It never covers the family
   being wrong. Step 4 is the counter-example to keep.
4. **`propagate` is not inference.** It draws priors and pushes couplings forward without ever
   calling your surrogate. `joint` conditions on the surrogate likelihoods, so information flows
   both ways along a coupling. The stored `inference_data.json` records which one produced it —
   read that field before you read the numbers.
5. **A green light can mean nothing happened — and so can a number.** You saw it as a skipped
   sweep in T2, as a preflight nobody branched on in this notebook's own first draft, as a
   posterior width from a direction the design never excited, and as a tight joint posterior
   from a chain that never moved. Always ask what success would look like if every step silently
   no-opped.

## Where to go next: `projects/tcr_signaling/`

If you found this exercise interesting, the **real** version of it lives in `projects/tcr_signaling/` — a git submodule reproducing Neve-Oz, Sherman & Raveh, "Bayesian metamodeling of early T-cell antigen receptor signaling," *Frontiers in Immunology* 2024. That submodule applies the same workflow you just did to **four real biological models**:

- Membrane topography (IRM-derived tight-contact geometry)
- Kinetic segregation (CD45 spatial exclusion from tight contacts)
- Lck activity (radial decay from CD45 boundaries)
- TCR phosphorylation (Lck* phosphorylates TCR ITAMs)

The spec contracts, the surrogate fits and the coupled metamodel look exactly like what you just built — **with the biology you came to do**. `specs/metamodel.tcr_signaling.json` is the four-model version of the spec you wrote in Step 6, and `notebooks/03_metamodel_inference.ipynb` is where it is sampled conditioned on four real fitted surrogates — the full-scale version of the two `meta sample` invocations you just compared. The submodule's `README.md` walks the reproduction. Note: it has its own dependencies (CMake, a C++ toolchain, and a Metal-only GPU kernel for the kinetic-segregation model) — see the parent `README.md` "Optional case study" section for what to expect.

There's no Tutorial 10. If you reach this point and still want more, the answer is: read a `projects/tcr_signaling/specs/*.json` file. They're the same shape you've been editing.

## Optional appendix: toolchain quality gate

The cells below run the package's repository-level quality gate (`ruff` + the fast test suite)
and two backend smoke tests. They are NOT part of the capstone — they check that your
*environment* is healthy, which is a different question from whether your pipeline is right.

A red result here is a statement about the framework in your environment, not about your
capstone: the fast suite covers `src/` and `tests/`, none of which you touched. Useful before
you push changes upstream, or if something earlier surprised you and you want to rule out the
install. Skip on a first read.

In [ ]:
# Quality gate scoped to the metamodeling package (src/ + tests/).
# `check=False` so a failing test or warning doesn't stop the notebook —
# the optional appendix is meant to be inspectable, not blocking.
run_tool("ruff", "format", "--check", "src", "tests", check=False)
run_tool("ruff", "check", "src", "tests", check=False)
run_tool("pytest", "-q", "-m", "not slow", "tests", check=False)


In [ ]:
if not PYMC_AVAILABLE:
    print("Optional backend smoke checks SKIPPED (PyMC missing in this kernel) — see preflight banner above.")
else:
    # Optional backend smoke checks (skipped cleanly if PyMC / SBI not installed).
    run_tool("pytest", "-q", "tests/test_surrogate_backends.py", "-k",
             "pymc_gp_backend_fit_sample_and_logprob", check=False)
    run_tool("pytest", "-q", "tests/test_surrogate_backends.py", "-k",
             "sbi_npe_backend_fit_sample_and_logprob", check=False)


## Final check: the capstone composed, and its lessons actually reproduced

This is the strictest self-check in the series, because a capstone that merely *ran* is worth
nothing. It asserts, in order:

1. **Step 1** — the store holds exactly `len(MY_GRID_A) × len(MY_GRID_B)` successful rows (no
   hardcoded 25; edit your grid freely) and **the loader hands the fit exactly those rows**, so
   the cumulative-store trap did not silently reopen.
2. **Step 2** — the artifact exists, records `backend=pymc_gp`, carries a spec digest and a
   dataset digest, and the fit retained a positive number of posterior draws.
3. **Step 3** — predictions match the analytical truth (`MAE < 1e-3`, six orders tighter than the
   old bound) **and the reported width brackets the error** (`MAE < 5 × mean width`), so a
   confidently-wrong surrogate fails rather than passes.
4. **Step 4** — the two failures actually reproduced: the rank-deficient DOE really is wrong by a
   visible margin, and the wrong-family fit really is wrong by many times its own reported width
   when extrapolated. If these pass silently, the notebook taught nothing.
5. **Step 6** — `propagate` left the coupling's source at its prior sd, `joint` moved *and*
   tightened it, `propagate` discarded the target's prior while `joint` respected it, and the
   chain that produced those numbers actually mixed on `a` and `b`.

Every bound has at least an order of magnitude of margin against the values this notebook
produces, so a refit on another machine will not trip it — but none of them is satisfiable by a
pipeline that skipped, stalled, or trained on the wrong rows.

In [ ]:
if not PYMC_AVAILABLE:
    print(f"\n[T9 self-check OK] PyMC steps skipped per preflight (PYMC_AVAILABLE={PYMC_AVAILABLE}).")
else:
    import csv as _csv
    import json as _json

    import numpy as _np

    from bayesian_metamodeling.spec import SurrogateSpec as _SurrogateSpec
    from bayesian_metamodeling.surrogates import eval_surrogate as _eval_surrogate
    from bayesian_metamodeling.surrogates.dataset import load_tabular_dataset as _load_dataset

    # --- 1. Step 1: the store holds YOUR DOE, and only YOUR DOE. ---
    _rows = []
    for _csv_path in sorted((root / CAPSTONE_STORE / "sweeps").rglob("sweep_rows.csv")):
        with open(_csv_path, newline="", encoding="utf-8") as _fh:
            _rows += [r for r in _csv.DictReader(_fh) if r.get("status") == "success"]
    assert len(_rows) == N_DOE, (
        f"Capstone store holds {len(_rows)} successful rows, expected {N_DOE} "
        f"({len(MY_GRID_A)}×{len(MY_GRID_B)}). A run store accumulates — Step 1 must clear it."
    )

    _spec = _SurrogateSpec.model_validate(
        _json.loads((root / "tmp/tutorials/specs/capstone.surrogate.pymc_gp.json").read_text())
    )
    _x_seen, _, _ = _load_dataset(_spec)
    assert _x_seen.shape[0] == N_DOE, (
        f"The loader hands the fit {_x_seen.shape[0]} rows, not your {N_DOE}."
    )

    # --- 2. Step 2: a real artifact with a real digest chain. ---
    _artifact = _json.loads((root / CAPSTONE_ART_PATH).read_text())
    assert _artifact["backend"] == "pymc_gp", f"backend={_artifact['backend']}"
    assert _artifact["spec_digest"] and _artifact["dataset_digest"], "artifact has no digests"

    _result = _eval_surrogate(spec=_spec, inputs_payload=QUERY_INPUTS, n=200)
    _posterior_draws = int(_result["summary"]["posterior_draws"])
    assert _posterior_draws > 0, "fit retained no posterior draws"

    # --- 3. Step 3: calibrated, and the band brackets the error. ---
    _mean = _np.asarray(_result["summary"]["mean"], dtype=float)
    _std = _np.asarray(_result["summary"]["std"], dtype=float)
    _truth = _np.asarray(QUERY_INPUTS["a"]) + _np.asarray(QUERY_INPUTS["b"])
    _mae = float(_np.mean(_np.abs(_mean - _truth)))
    assert _mae < 1e-3, (
        f"MAE={_mae:.3e} on an exactly-linear truth with a linear surrogate. Either the fit is "
        f"broken, or your DOE does not identify both weights (try varying BOTH a and b)."
    )
    assert _mae < 5.0 * float(_np.mean(_std)), (
        f"MAE={_mae:.3e} exceeds 5x the reported width {float(_np.mean(_std)):.3e} — the "
        f"surrogate is confidently wrong, which is worse than being uncertain."
    )

    # --- 4. Step 4: the two failures reproduced, so the lesson is not decorative. ---
    _flat = BREAKAGE["a never varies, y = a+b"]
    assert _flat["mae"] > 0.05, (
        f"The rank-deficient DOE scored MAE={_flat['mae']:.3e}; it is supposed to FAIL. "
        f"If it succeeded, Step 4 demonstrated nothing."
    )
    assert FAR_RATIO > 5.0, (
        f"Extrapolating the wrong-family fit was only {FAR_RATIO:.1f}x its own reported width "
        f"off the truth; the 'a band does not cover a wrong family' lesson did not reproduce."
    )

    # --- 5. Step 6: propagate propagated, joint inferred, and the chain moved. ---
    _meta = META_RESULTS["tutorial9_capstone_meta"]
    _prop_a, _joint_a = _meta["prop"]["a"], _meta["joint"]["a"]
    _prop_sd, _joint_sd = float(_prop_a.std(ddof=1)), float(_joint_a.std(ddof=1))
    assert abs(_prop_sd - PRIOR_A[1]) < 0.1 * PRIOR_A[1], (
        f"propagate returned sd(a)={_prop_sd:.3f} but a's prior sd is {PRIOR_A[1]}. propagate "
        f"draws the coupling SOURCE straight from its prior; if this moved, it is not propagate."
    )
    assert _joint_sd < 0.95 * _prop_sd, (
        f"joint sd(a)={_joint_sd:.3f} did not tighten against propagate's {_prop_sd:.3f} — the "
        f"surrogate likelihood carried no information back to the inputs."
    )
    assert abs(float(_joint_a.mean()) - PRIOR_A[0]) > 0.15, (
        f"joint mean(a)={float(_joint_a.mean()):.3f} barely left the prior mean {PRIOR_A[0]}; "
        f"the measurement did not reach `a` through the surrogate."
    )
    _p_obs = float(_meta["prop"]["y_obs"].std(ddof=1))
    _j_obs = float(_meta["joint"]["y_obs"].std(ddof=1))
    assert _p_obs > 3.0 * _j_obs, (
        f"propagate sd(y_obs)={_p_obs:.3f} vs joint {_j_obs:.3f}: propagate is supposed to "
        f"DISCARD the prior on a coupling target and overwrite it."
    )
    _stuck = _meta["info"].get("poorly_mixed") or []
    assert "a" not in _stuck and "b" not in _stuck, (
        f"The joint chain barely moved on {_stuck}: its posterior summaries are meaningless. "
        f"Your DOE produced a surrogate too sharp to condition on — widen the grid so the "
        f"a·b fit has honest residual width."
    )

    print(
        f"\n[T9 self-check OK] DOE {len(_rows)}/{N_DOE} rows (loader saw {_x_seen.shape[0]}); "
        f"MAE={_mae:.2e} < 1e-3 and < 5x width {float(_np.mean(_std)):.2e}; "
        f"rank-deficient DOE failed as designed (MAE={_flat['mae']:.3f}); "
        f"wrong-family extrapolation off by {FAR_RATIO:.0f}x its own width; "
        f"sd(a) prior {PRIOR_A[1]:.3f} -> propagate {_prop_sd:.3f} -> joint {_joint_sd:.3f}; "
        f"pipeline composes spec -> sweep -> surrogate -> metamodel end-to-end."
    )